# Synthetic Demonstration: Study-Grouped Nanotoxicity Classification

This self-contained notebook demonstrates the leakage-safe classification pattern used in the full project. It creates synthetic observations with the same broad structure as the scientific workflow.

**Important:** synthetic values and metrics are not scientific findings. They exist only so reviewers can run and inspect the pipeline without the source spreadsheets.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, average_precision_score
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)


## Create structurally representative synthetic data

Each record includes particle, exposure, assay, cell-system and source-study fields. The endpoint is generated for demonstration only.


In [ ]:
n = 1800
n_studies = 90

df = pd.DataFrame({
    "Pubmed ID": rng.integers(100000, 100000 + n_studies, n).astype(str),
    "Material type": rng.choice(["ZnO", "TiO2", "CuO", "Fe2O3", "CeO2"], n),
    "Core size (nm)": rng.lognormal(3.2, 0.55, n),
    "Hydro size (nm)": rng.lognormal(4.1, 0.60, n),
    "Surface charge (mV)": rng.normal(-8, 18, n),
    "Mass dose (ug/mL)": rng.lognormal(2.6, 1.0, n),
    "Exposure time (hours)": rng.choice([6, 12, 24, 48, 72], n),
    "Assay": rng.choice(["MTT", "WST-1", "LDH", "Alamar Blue"], n),
    "Cell type": rng.choice(["epithelial", "macrophage", "fibroblast"], n),
})

material_effect = df["Material type"].map(
    {"ZnO": 0.7, "TiO2": -0.4, "CuO": 1.0, "Fe2O3": -0.2, "CeO2": -0.3}
)
linear_risk = (
    -3.2
    + 0.55 * np.log1p(df["Mass dose (ug/mL)"])
    + 0.012 * df["Exposure time (hours)"]
    - 0.012 * df["Surface charge (mV)"]
    + material_effect
    + rng.normal(0, 0.65, n)
)
p_toxic = 1 / (1 + np.exp(-linear_risk))
df["Toxicity"] = np.where(rng.random(n) < p_toxic, "Toxic", "Nontoxic")
df["Viability (%)"] = np.clip(
    82 - 38 * (df["Toxicity"] == "Toxic") + rng.normal(0, 12, n), 0, 140
)

print(df.shape)
print(df["Toxicity"].value_counts())
print(df["Toxicity"].value_counts(normalize=True).round(3))
df.head()


## Define a leakage-safe problem

`Viability (%)` is excluded because it directly informs the toxicity label. `Pubmed ID` is used only to define validation groups.


In [ ]:
numeric_features = [
    "Core size (nm)", "Hydro size (nm)", "Surface charge (mV)",
    "Mass dose (ug/mL)", "Exposure time (hours)"
]
categorical_features = ["Material type", "Assay", "Cell type"]
features = numeric_features + categorical_features

X = df[features].copy()
y = (df["Toxicity"] == "Toxic").astype(int)
groups = df["Pubmed ID"]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5)),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipe, numeric_features),
    ("categorical", categorical_pipe, categorical_features),
])

models = {
    "Balanced Logistic Regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced_subsample",
        min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1
    ),
}


## Five-fold stratified group validation

No synthetic PubMed study is allowed in both the training and validation partitions of a fold. Preprocessing is fitted inside each fold.


In [ ]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []

for model_name, model in models.items():
    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):
        pipeline = Pipeline([
            ("preprocess", preprocessor),
            ("model", model),
        ])
        pipeline.fit(X.iloc[train_idx], y.iloc[train_idx])
        pred = pipeline.predict(X.iloc[test_idx])
        prob = pipeline.predict_proba(X.iloc[test_idx])[:, 1]

        rows.append({
            "Model": model_name,
            "Fold": fold,
            "Accuracy": accuracy_score(y.iloc[test_idx], pred),
            "Balanced Accuracy": balanced_accuracy_score(y.iloc[test_idx], pred),
            "Precision": precision_score(y.iloc[test_idx], pred, zero_division=0),
            "Recall": recall_score(y.iloc[test_idx], pred, zero_division=0),
            "F1": f1_score(y.iloc[test_idx], pred, zero_division=0),
            "PR-AUC": average_precision_score(y.iloc[test_idx], prob),
        })

results = pd.DataFrame(rows)
results.groupby("Model")[
    ["Accuracy", "Balanced Accuracy", "Precision", "Recall", "F1", "PR-AUC"]
].agg(["mean", "std"]).round(3)


## Interpretation

The demonstration shows the engineering pattern, not the reported scientific performance. The portfolio's actual project results came from the supplied literature-derived datasets and are summarized in the repository README.
